# GridSketcher: a conformal regional grid from a sketched outline

`GridSketcher` builds an orthogonal (conformal) regional `Grid` from a hand-drawn coastline outline and four chosen corners, instead of a bounding box or a projected extent. It is a good fit whenever the domain boundary should follow an irregular coastline rather than straight lon/lat or projected edges.

The same solve is also available without the widget, as `Grid.from_outline` -- see "Building a grid directly from an outline" below. Either way the result is a plain `Grid` that remembers the outline it came from, so the two are interchangeable: open `GridSketcher(grid)` on a grid built by `Grid.from_outline` to keep refining it, or call `Grid.from_outline` once you are happy with a sketch built interactively.

This notebook is a short walkthrough of the widget. Run it in JupyterLab: it needs a live, interactive `ipympl` frontend, and the docs build does not execute notebooks.


## 1. Launch the sketcher

With no arguments, `GridSketcher` starts from a small default box (the CrocoDash tutorial domain). `resolution_km` sets the nominal (mean) cell size.

In [ ]:
%matplotlib ipympl
from mom6_forge.grid_sketcher import GridSketcher

sketch = GridSketcher(resolution_km=10)
sketch

## 2. Sketch the outline

On the map (left), as the box under it also says:

- **Drag** a vertex to move it.
- **Click an edge** to insert a new vertex there.
- **Right-click** (or **Ctrl-click**, for a one-button trackpad) a vertex to delete it -- the remaining vertices keep their corner status.
- **Double-click** a vertex to make or unmake it one of the outline's 4 corners (numbered 1-4 on the map).
- **Scroll** inside the framed map to zoom about the cursor (out to the whole hemisphere facing you); **drag** the empty map (or middle-drag anywhere) to pan; **Reset view** under the map shows the whole outline again.
- **Clear all** to draw a new outline: click points (drag one to move it, right-click one to delete it), then click point 1 again to close it (4 points become the 4 corners). While drawing, a pan turns the globe to face the middle of the view, so any place on Earth can be reached.
- Hover over the grid to see why a line is colored: grey/orange/red mean ok/warn/bad orthogonality, size ratio, or aspect ratio. Only ocean cells are judged (from the coastline). Rest the cursor on a grid cell and a box there explains it (or says it is over land); off the grid no box appears.

On the panel (right): **Resolution (km)**, the **Projection** buttons (the projection the grid is solved in, not the map's; the pressed one is in use and named in the summary line; the green one is recommended, as its cells vary least in size, and a new sketch or newly drawn outline is solved on it until you press another), **Shade cell size**, **Undo**/**Redo**, and the diagnostics shown in **Details**. Under **Details** and **Edit coordinates**, **Errors** say why **Build final grid** is greyed out (fewer than 4 corners, crossing edges, a failed solve, folded ocean cells) and **Warnings** list what to check (abrupt size changes, non-orthogonal or stretched cells, nearly flat corners, short or coast-hugging open boundaries), one line each; any warning turns Build amber.

## 3. Build and save the grid

**Build final grid** solves the outline at full resolution and stores the result as `sketch.grid` -- an ordinary `mom6_forge.grid.Grid`. Like `Grid.from_outline` below, it carries a `.outline` attribute, so `GridSketcher(sketch.grid)` reopens the same outline for further editing.


In [ ]:
# After clicking "Build final grid" in the panel above:
grid = sketch.grid
grid

**Save** writes the built grid to `GridLibrary/grid_<name>.nc`. The file keeps the outline, so `GridSketcher(Grid.from_supergrid(path))` reopens the sketch.

## 4. Use the grid like any other

`grid` is a regular `Grid`: pass it to `Topo` (and to CrocoDash's `Case`) exactly as with a grid built from `Grid.from_projection`, `Grid.from_center`, or `Grid.from_outline`.


In [ ]:
from mom6_forge.topo import Topo

topo = Topo(grid, min_depth=10.0)

## 5. Starting from an existing grid

`GridSketcher(grid)` starts a new sketch from an existing grid. If the grid has an `.outline` (built by `Grid.from_outline`, or by a previous "Build final grid"), that outline is reopened exactly as it was. Any other grid has its own boundary traced, with `per_side` vertices per side (default 1, a plain 4-vertex outline); use more for a boundary that isn't a straight line in the sketching projection, such as a lon/lat box -- useful for re-sketching a domain someone already built.


In [ ]:
from mom6_forge.grid import Grid

box = Grid(lenx=8.0, leny=8.0, resolution=0.1, xstart=235.0, ystart=31.0, name="box")
resketch = GridSketcher(box, per_side=4)  # box has no .outline: its edge is traced
resketch

## 6. Building a grid directly from an outline

`Grid.from_outline` is `GridSketcher`'s solve without the widget: give it an outline (>= 4 vertices, in degrees) and the 4 corner indices that mark the grid's corners (defaulted to `[0, 1, 2, 3]` for a plain 4-vertex outline, required otherwise) and it returns a `Grid` directly -- handy in a script, or once a sketched outline's coordinates are already known.

Set either `resolution_km` (the nominal, mean cell size -- the same quantity the widget's **Resolution (km)** control sets) or `n_cells` (target `nx * ny`), and `projection` (a kind name such as `"lcc"`, a `MapProjection` from `mom6_forge.grid_sketcher`, or `"auto"`, the default, which picks the widget's recommendation: the kind whose cells vary least in size).


In [ ]:
from mom6_forge.grid import Grid

lon = [235.0, 243.0, 243.0, 235.0]
lat = [31.0, 31.0, 39.0, 39.0]
direct_grid = Grid.from_outline(lon, lat, resolution_km=10, name="direct")
direct_grid

`direct_grid.outline` records the outline, corners, resolved projection, and resolution it was built from, so it can still be opened in the widget to keep refining it interactively:


In [ ]:
%matplotlib ipympl
from mom6_forge.grid_sketcher import GridSketcher

sketch2 = GridSketcher(direct_grid)  # reopens direct_grid.outline exactly
sketch2